# 02 · Run the spatial overview

Run or resume the complete local spatial overview. On this accepted AB workspace, the standard `02_cellcharter` outputs are already present, so the notebook leaves them unchanged and generates only missing nncomp or directional-overlap outputs. A new user without those outputs follows the same notebook and runs a genuine fresh CellCharter stage because the active YAML contains no reuse mapping.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

from spatial_workflow.config import load_config
from spatial_workflow.overview import run_spatial_overview
from spatial_workflow.launchers import (
    write_spatial_overview_script,
    write_tmux_launcher,
)

CONFIG_PATH = REPO_ROOT / "configs" / "local.yaml"
JOB_PATH = REPO_ROOT / "jobs" / "02_spatial_overview.sh"
TMUX_PATH = REPO_ROOT / "jobs" / "02_spatial_overview_tmux.sh"
LOG_PATH = REPO_ROOT / "logs" / "02_spatial_overview.log"

config = load_config(CONFIG_PATH)
config["nncomp"]

{'input_h5ad': '02_cellcharter/ab_xenium_cellcharter.h5ad',
 'output_dir': '03_nncomp',
 'graph': {'method': 'squidpy_knn', 'n_neighs': 6, 'radius': 30},
 'overlap': {'enabled': True,
  'k_values': [1, 2, 6],
  'analyses': ['whole_sample', 'within_compartment', 'compartment_pair'],
  'source_cell_types': ['OLIG_sub7'],
  'neighbor_cell_types': None,
  'n_permutations': 1000,
  'seed': 0,
  'workers': -1,
  'permutation_plan': {'seed': 0,
   'endpoint_roles': ['shared', 'source', 'target']},
  'reciprocal_cell_types': {'enabled': True,
   'n_permutations': 1000,
   'seed': 0,
   'workers': -1,
   'progress_every': 100}}}

Run the resume-aware overview now. Complete stage outputs are never overwritten, partial stages fail loudly, and missing stages are generated in order. Here that means the copied/local CellCharter contract is treated as complete, the accepted base nncomp tables are retained, and only the three missing overlap Parquets are calculated.

In [2]:
overview = run_spatial_overview(CONFIG_PATH)
print("Actions:", ", ".join(overview["actions"]))
for stage in (
    "cellcharter",
    "nncomp",
    "directional_overlap",
    "permutation_plan",
    "reciprocal_celltypes",
):
    for name, path in overview[stage].items():
        print(f"{stage}.{name}: {path}")

Actions: used_existing_cellcharter, used_existing_nncomp, used_existing_directional_overlap, used_existing_permutation_plan, used_existing_reciprocal_celltype_permutation
cellcharter.h5ad: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/02_cellcharter/ab_xenium_cellcharter.h5ad
cellcharter.summary: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/02_cellcharter/ab_xenium_cellcharter.summary.json
cellcharter.abundance: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/02_cellcharter/ab_xenium_cellcharter.domain_abundance.csv
cellcharter.stability: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/02_cellcharter/ab_xenium_cellcharter.autok_stability.csv
nncomp.compartment_abundance: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/03_nncomp/compartment_abundance.csv
nncomp.neighbor_composition: /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/results/ab_xenium/03_nncomp/neighbor_compositi

Generate the same resume-aware shell job and, optionally, a small `tmux` launcher. On a clean workspace the job runs CellCharter and nncomp; on this workspace it skips the completed local stages.

In [3]:
job_path = write_spatial_overview_script(CONFIG_PATH, JOB_PATH)
print(job_path.read_text())

WRITE_TMUX_LAUNCHER = False

if WRITE_TMUX_LAUNCHER:
    tmux_path = write_tmux_launcher(
        job_script=job_path,
        output_path=TMUX_PATH,
        session_name=config.get("runtime", {}).get(
            "tmux_session", "spatial_overview"
        ),
        log_path=LOG_PATH,
    )
    print(tmux_path.read_text())
    print(f"Run on the compute node when ready: bash {tmux_path}")
else:
    print("Set WRITE_TMUX_LAUNCHER = True to write the optional launcher.")

#!/usr/bin/env bash
set -euo pipefail

/stor/home/ncr828/cci_venv/bin/python3 -m spatial_workflow.overview --config /stor/scratch/WCAAR/rhyan_scratch/spatial-workflow/configs/local.yaml

Set WRITE_TMUX_LAUNCHER = True to write the optional launcher.
